In [1]:
import os
import pandas as pd

import cobra
from cobra.core import Configuration
from cobra.io import load_json_model
from cobra.flux_analysis import single_gene_deletion
from cobra.manipulation import knock_out_model_genes

In [2]:
os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

# Growth rate comparison between models

In [3]:
# Load the input model of Salmonella Tm. - stm_v1_0
stm_v1 = load_json_model("STM_v1_0.json")

# Load the extracted Salmonella models. 
# Mixed - 278 sample of cs/bth/stm
# Stm - 279 sample of Salmonella
mixed1090 = load_json_model("run_031526/context_models/mixed1090_imat_restrictions.json")
mixed1585 = load_json_model("run_031526/context_models/mixed1585_imat_restrictions.json")
mixed2575 = load_json_model("run_031526/context_models/mixed2575_imat_restrictions.json")
stm1090 = load_json_model("run_031526/context_models/stm1090_imat_restrictions.json")
stm1585 = load_json_model("run_031526/context_models/stm1585_imat_restrictions.json")
stm2575 = load_json_model("run_031526/context_models/stm2575_imat_restrictions.json")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2789960
Academic license 2789960 - for non-commercial use only - registered to gm___@ucdavis.edu


In [4]:
models = {
    "input_stm_v1": stm_v1,
    "mixed_10_90": mixed1090,
    "mixed_15_85": mixed1585,
    "mixed_25_75": mixed2575,
    "stm_10_90": stm1090,
    "stm_15_85": stm1585,
    "stm_25_75": stm2575,
}

In [5]:
fba_results = []

for model_name, model in models.items():
    solution = model.optimize()

    fba_results.append({
        "model": model_name,
        "status": solution.status,
        "objective_value": solution.objective_value,
        "n_reactions": len(model.reactions),
        "n_metabolites": len(model.metabolites),
        "n_genes": len(model.genes),
    })

fba_df = pd.DataFrame(fba_results)

display(fba_df)

,model,status,objective_value,n_reactions,n_metabolites,n_genes
0,input_stm_v1,optimal,0.477834,2545,1802,1271
1,mixed_10_90,optimal,0.477834,2545,1802,1271
2,mixed_15_85,optimal,0.477834,2545,1802,1271
3,mixed_25_75,optimal,0.477834,2545,1802,1271
4,stm_10_90,optimal,0.477834,2545,1802,1271
5,stm_15_85,optimal,0.477834,2545,1802,1271
6,stm_25_75,optimal,0.477834,2545,1802,1271


Make a bar chart comparing the objective value of all models. 

In [6]:
# The difference in objective growth when compared to the input model
input_growth = fba_df.loc[
    fba_df["model"] == "input_stm_v1",
    "objective_value"
].iloc[0]

fba_df["difference_vs_input"] = fba_df["objective_value"] - input_growth
fba_df["fraction_of_input"] = fba_df["objective_value"] / input_growth

display(fba_df.sort_values("fraction_of_input"))

fba_df.to_csv("fba_objective_values_all_models.csv", index=False)

,model,status,objective_value,n_reactions,n_metabolites,n_genes,difference_vs_input,fraction_of_input
0,input_stm_v1,optimal,0.477834,2545,1802,1271,0.0,1.0
1,mixed_10_90,optimal,0.477834,2545,1802,1271,0.0,1.0
2,mixed_15_85,optimal,0.477834,2545,1802,1271,0.0,1.0
3,mixed_25_75,optimal,0.477834,2545,1802,1271,0.0,1.0
4,stm_10_90,optimal,0.477834,2545,1802,1271,0.0,1.0
5,stm_15_85,optimal,0.477834,2545,1802,1271,0.0,1.0
6,stm_25_75,optimal,0.477834,2545,1802,1271,0.0,1.0


# Flux Through Key Pathways

Let's look at the flux of reactions with the following genes: 

TCA: sdhA/B/C/D  
Anaerobic respiration: frdA/B/C/D  
Fermentation: ldhA, dld  
Respiration: cyoA/B/C/D, cydA/B  

In [7]:
target_genes = ["sdh", "frd", "ldh", "dld", "cyo", "cyd"]

rows = []

for name, model in models.items():
    sol = model.optimize()
    
    for rxn in model.reactions:
        text = (rxn.id + rxn.name + rxn.gene_reaction_rule).lower()
        
        if any(k in text for k in target_genes):
            rows.append({
                "model": name,
                "reaction": rxn.id,
                "flux": sol.fluxes[rxn.id]
            })

flux_df = pd.DataFrame(rows)
display(flux_df)

,model,reaction,flux
0,input_stm_v1,DHFR,0.036381
1,input_stm_v1,FRD2,0.205061
2,input_stm_v1,FRD3,0.000000
3,input_stm_v1,LDH_D,0.000000
4,input_stm_v1,LDH_D2,0.000000
...,...,...,...
65,stm_25_75,PALDH,0.000000
66,stm_25_75,FALDH2,0.000000
67,stm_25_75,TSULDHpp1,0.000000
68,stm_25_75,TSULDHpp2,0.000000


In [8]:
# Run FBA on the input model only

solution = stm_v1.optimize()

print("Status:", solution.status)
print("Objective value:", solution.objective_value)

Status: optimal
Objective value: 0.47783366076072126


In [9]:
# Collect all reaction fluxes from stm_v1

flux_df = solution.fluxes.reset_index()
flux_df.columns = ["reaction_id", "flux"]

reaction_info = []

for rxn in stm_v1.reactions:
    reaction_info.append({
        "reaction_id": rxn.id,
        "reaction_name": rxn.name,
        "reaction_equation": rxn.reaction,
        "gpr": rxn.gene_reaction_rule,
        "lower_bound": rxn.lower_bound,
        "upper_bound": rxn.upper_bound,
    })

reaction_info_df = pd.DataFrame(reaction_info)

flux_annotated = reaction_info_df.merge(
    flux_df,
    on="reaction_id",
    how="left"
)

display(flux_annotated.head())

,reaction_id,reaction_name,reaction_equation,gpr,lower_bound,upper_bound,flux
0,3HAD180,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,3hoctaACP_c --> h2o_c + toctd2eACP_c,STM1067 or STM0227,0.0,1000.0,0.002079
1,3HAD181,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,3hcvac11eACP_c --> h2o_c + t3c11vaceACP_c,STM0227 or STM1067,0.0,1000.0,0.003118
2,3HAD40,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,3haACP_c --> but2eACP_c + h2o_c,STM1067 or STM0227,0.0,1000.0,0.105576
3,3HAD60,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,3hhexACP_c --> h2o_c + thex2eACP_c,STM1067 or STM0227,0.0,1000.0,0.105576
4,3HAD80,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,3hoctACP_c --> h2o_c + toct2eACP_c,STM1067 or STM0227,0.0,1000.0,0.105576


In [10]:
# Define targeted metabolic systems for sanity-check interpretation

keywords = [
    "sdh",   # succinate dehydrogenase / TCA
    "frd",   # fumarate reductase / anaerobic respiration
    "ldh",   # lactate dehydrogenase
    "dld",   # D-lactate dehydrogenase
    "cyo",   # cytochrome bo oxidase
    "cyd",   # cytochrome bd oxidase
    "pfl",   # pyruvate formate lyase
    "pta",   # phosphate acetyltransferase
    "ack",   # acetate kinase
    "adh",   # alcohol dehydrogenase
    "nar",   # nitrate reductase
    "nap",   # periplasmic nitrate reductase
    "nir",   # nitrite reductase
]

In [11]:
# Filter reactions whose ID, name, equation, or GPR matches the keywords

def matches_keywords(row):
    text = " ".join([
        str(row["reaction_id"]),
        str(row["reaction_name"]),
        str(row["reaction_equation"]),
        str(row["gpr"]),
    ]).lower()
    
    return any(k in text for k in keywords)

target_fluxes = flux_annotated[
    flux_annotated.apply(matches_keywords, axis=1)
].copy()

target_fluxes["abs_flux"] = target_fluxes["flux"].abs()

target_fluxes = target_fluxes.sort_values(
    "abs_flux",
    ascending=False
)

display(target_fluxes)

,reaction_id,reaction_name,reaction_equation,gpr,lower_bound,upper_bound,flux,abs_flux
1568,NADH16pp,NADH dehydrogenase (ubiquinone-8 & 3 protons) ...,4.0 h_c + nadh_c + q8_c --> 3.0 h_p + nad_c + ...,STM2316_S and STM2317 and STM2318 and STM2319 ...,0.0,1000.0,19.706532,19.706532
1057,GAPD,Glyceraldehyde-3-phosphate dehydrogenase,g3p_c + nad_c + pi_c <=> 13dpg_c + h_c + nadh_c,STM1290,-1000.0,1000.0,8.369636,8.369636
1704,PDH,Pyruvate dehydrogenase,coa_c + nad_c + pyr_c --> accoa_c + co2_c + na...,STM0152 and STM0153 and STM0154,0.0,1000.0,4.403788,4.403788
1411,MDH,Malate dehydrogenase,mal__L_c + nad_c <=> h_c + nadh_c + oaa_c,STM3359,-1000.0,1000.0,3.392388,3.392388
257,AKGDH,2-Oxogluterate dehydrogenase,akg_c + coa_c + nad_c --> co2_c + nadh_c + suc...,STM0154 and STM0736 and STM0737,0.0,1000.0,2.971215,2.971215
...,...,...,...,...,...,...,...,...
2482,GULNLR,L-gulonate reductase,guln__L_c + 2.0 nad_c --> fruur_c + 2.0 nadh_c,,0.0,1000.0,0.000000,0.000000
2483,HHDDI,"2-hydroxyhepta-2,4-diene-1,7-dioate isomerase",2hh24dd_c <=> 2ohed_c,STM1101,-1000.0,1000.0,0.000000,0.000000
2491,MMSAD3,Methylmalonate-semialdehyde dehydrogenase (mal...,coa_c + msa_c + nad_c --> accoa_c + co2_c + na...,STM4421,0.0,1000.0,0.000000,0.000000
2499,FEENTERR1,Fe-enterobactin reduction (Fe(III)-unloading),fadh2_c + 2.0 feenter_c --> 2.0 enter_c + fad_...,STM0586,0.0,1000.0,0.000000,0.000000


In [12]:
# Show only reactions carrying nonzero flux

active_target_fluxes = target_fluxes[
    target_fluxes["abs_flux"] > 1e-9
].copy()

display(active_target_fluxes)

,reaction_id,reaction_name,reaction_equation,gpr,lower_bound,upper_bound,flux,abs_flux
1568,NADH16pp,NADH dehydrogenase (ubiquinone-8 & 3 protons) ...,4.0 h_c + nadh_c + q8_c --> 3.0 h_p + nad_c + ...,STM2316_S and STM2317 and STM2318 and STM2319 ...,0.0,1000.0,19.706532,19.706532
1057,GAPD,Glyceraldehyde-3-phosphate dehydrogenase,g3p_c + nad_c + pi_c <=> 13dpg_c + h_c + nadh_c,STM1290,-1000.0,1000.0,8.369636,8.369636
1704,PDH,Pyruvate dehydrogenase,coa_c + nad_c + pyr_c --> accoa_c + co2_c + na...,STM0152 and STM0153 and STM0154,0.0,1000.0,4.403788,4.403788
1411,MDH,Malate dehydrogenase,mal__L_c + nad_c <=> h_c + nadh_c + oaa_c,STM3359,-1000.0,1000.0,3.392388,3.392388
257,AKGDH,2-Oxogluterate dehydrogenase,akg_c + coa_c + nad_c --> co2_c + nadh_c + suc...,STM0154 and STM0736 and STM0737,0.0,1000.0,2.971215,2.971215
340,ASPTA,Aspartate transaminase,akg_c + asp__L_c <=> glu__L_c + oaa_c,STM0998,-1000.0,1000.0,-1.320155,1.320155
1734,PGCD,Phosphoglycerate dehydrogenase,3pg_c + nad_c --> 3php_c + h_c + nadh_c,STM3062,0.0,1000.0,1.219064,1.219064
2437,BIOMASS_iRR1083_metals,Biomass equation from http://www.biomedcentral...,0.000188 12dgr2_ST_p + 0.05 5mthf_c + 5e-05 ac...,,0.0,1000.0,0.477834,0.477834
1017,FRD2,Fumarate reductase,fum_c + mql8_c --> mqn8_c + succ_c,STM4340 and STM4341 and STM4342 and STM4343,0.0,1000.0,0.205061,0.205061
1994,SDPTA,Succinyldiaminopimelate transaminase,akg_c + sl26da_c <=> glu__L_c + sl2a6o_c,STM3468,-1000.0,1000.0,-0.172406,0.172406


In [13]:
# Save results

target_fluxes.to_csv("simulation_2_stm_v1_target_pathway_fluxes_all.csv", index=False)
active_target_fluxes.to_csv("simulation_2_stm_v1_target_pathway_fluxes_active.csv", index=False)

In [14]:
display(
    active_target_fluxes[
        [
            "reaction_id",
            "reaction_name",
            "flux",
            "reaction_equation",
            "gpr",
        ]
    ].head(20)
)

,reaction_id,reaction_name,flux,reaction_equation,gpr
1568,NADH16pp,NADH dehydrogenase (ubiquinone-8 & 3 protons) ...,19.706532,4.0 h_c + nadh_c + q8_c --> 3.0 h_p + nad_c + ...,STM2316_S and STM2317 and STM2318 and STM2319 ...
1057,GAPD,Glyceraldehyde-3-phosphate dehydrogenase,8.369636,g3p_c + nad_c + pi_c <=> 13dpg_c + h_c + nadh_c,STM1290
1704,PDH,Pyruvate dehydrogenase,4.403788,coa_c + nad_c + pyr_c --> accoa_c + co2_c + na...,STM0152 and STM0153 and STM0154
1411,MDH,Malate dehydrogenase,3.392388,mal__L_c + nad_c <=> h_c + nadh_c + oaa_c,STM3359
257,AKGDH,2-Oxogluterate dehydrogenase,2.971215,akg_c + coa_c + nad_c --> co2_c + nadh_c + suc...,STM0154 and STM0736 and STM0737
340,ASPTA,Aspartate transaminase,-1.320155,akg_c + asp__L_c <=> glu__L_c + oaa_c,STM0998
1734,PGCD,Phosphoglycerate dehydrogenase,1.219064,3pg_c + nad_c --> 3php_c + h_c + nadh_c,STM3062
2437,BIOMASS_iRR1083_metals,Biomass equation from http://www.biomedcentral...,0.477834,0.000188 12dgr2_ST_p + 0.05 5mthf_c + 5e-05 ac...,
1017,FRD2,Fumarate reductase,0.205061,fum_c + mql8_c --> mqn8_c + succ_c,STM4340 and STM4341 and STM4342 and STM4343
1994,SDPTA,Succinyldiaminopimelate transaminase,-0.172406,akg_c + sl26da_c <=> glu__L_c + sl2a6o_c,STM3468


# Gene Deletions

In [15]:
# Run single-gene deletion on stm_v1

gene_del = single_gene_deletion(stm_v1)

display(gene_del.head())

,ids,growth,status
0,{STM2991},0.477834,optimal
1,{STM0255},0.477834,optimal
2,{STM2256},0.477834,optimal
3,{STM2064},0.477834,optimal
4,{STM4453},0.477834,optimal


In [16]:
# Clean up the gene deletion table

gene_del_df = gene_del.copy()

gene_del_df["gene_id"] = gene_del_df["ids"].apply(lambda x: list(x)[0])
gene_del_df = gene_del_df.drop(columns=["ids"])

gene_del_df["growth_fraction"] = gene_del_df["growth"] / solution.objective_value

gene_del_df["essential"] = gene_del_df["growth_fraction"] < 0.01
gene_del_df["reduced_growth"] = (
    (gene_del_df["growth_fraction"] >= 0.01) &
    (gene_del_df["growth_fraction"] < 0.5)
)

gene_del_df = gene_del_df.sort_values("growth_fraction")

display(gene_del_df.head(30))

,growth,status,gene_id,growth_fraction,essential,reduced_growth
1220,0.0,optimal,STM2511,0.0,True,False
1196,0.0,optimal,STM0171,0.0,True,False
1199,0.0,optimal,STM3200,0.0,True,False
74,0.0,optimal,STM1197,0.0,True,False
75,0.0,optimal,STM2817,0.0,True,False
520,0.0,optimal,STM1777,0.0,True,False
521,0.0,optimal,STM2072,0.0,True,False
633,0.0,optimal,STM2946,0.0,True,False
80,0.0,optimal,STM1426,0.0,True,False
1207,0.0,optimal,STM1723,0.0,True,False


In [17]:
# Add gene names from the model

gene_info = pd.DataFrame({
    "gene_id": [gene.id for gene in stm_v1.genes],
    "gene_name": [gene.name for gene in stm_v1.genes],
})

gene_del_annotated = gene_del_df.merge(
    gene_info,
    on="gene_id",
    how="left"
)

display(gene_del_annotated.head(30))

,growth,status,gene_id,growth_fraction,essential,reduced_growth,gene_name
0,0.0,optimal,STM2511,0.0,True,False,guaB
1,0.0,optimal,STM0171,0.0,True,False,yadF
2,0.0,optimal,STM3200,0.0,True,False,rfaE
3,0.0,optimal,STM1197,0.0,True,False,fabF
4,0.0,optimal,STM2817,0.0,True,False,luxS
5,0.0,optimal,STM1777,0.0,True,False,hemA
6,0.0,optimal,STM2072,0.0,True,False,hisD
7,0.0,optimal,STM2946,0.0,True,False,cysH
8,0.0,optimal,STM1426,0.0,True,False,ribE
9,0.0,optimal,STM1723,0.0,True,False,trpE


In [18]:
# Count essential and reduced-growth genes

summary = pd.DataFrame({
    "category": [
        "essential",
        "reduced_growth",
        "nonessential"
    ],
    "n_genes": [
        gene_del_annotated["essential"].sum(),
        gene_del_annotated["reduced_growth"].sum(),
        ((gene_del_annotated["essential"] == False) &
         (gene_del_annotated["reduced_growth"] == False)).sum()
    ]
})

display(summary)

,category,n_genes
0,essential,202
1,reduced_growth,8
2,nonessential,1061


In [19]:
# Show only essential genes

essential_genes = gene_del_annotated[
    gene_del_annotated["essential"]
].copy()

display(
    essential_genes[
        [
            "gene_id",
            "gene_name",
            "growth",
            "growth_fraction",
            "status"
        ]
    ]
)

,gene_id,gene_name,growth,growth_fraction,status
0,STM2511,guaB,0.000000e+00,0.000000e+00,optimal
1,STM0171,yadF,0.000000e+00,0.000000e+00,optimal
2,STM3200,rfaE,0.000000e+00,0.000000e+00,optimal
3,STM1197,fabF,0.000000e+00,0.000000e+00,optimal
4,STM2817,luxS,0.000000e+00,0.000000e+00,optimal
...,...,...,...,...,...
197,STM4139,coaA,9.123545e-12,1.909356e-11,optimal
198,STM0181,panC,9.123545e-12,1.909356e-11,optimal
199,STM0180,panD,9.523273e-12,1.993010e-11,optimal
200,STM3730,dfp,1.971234e-11,4.125357e-11,optimal


In [20]:
# Show genes that strongly reduce growth but are not fully essential

reduced_growth_genes = gene_del_annotated[
    gene_del_annotated["reduced_growth"]
].copy()

display(
    reduced_growth_genes[
        [
            "gene_id",
            "gene_name",
            "growth",
            "growth_fraction",
            "status"
        ]
    ].head(30)
)

,gene_id,gene_name,growth,growth_fraction,status
202,STM3869,atpF,0.115896,0.242544,optimal
203,STM3864,atpC,0.115896,0.242544,optimal
204,STM3865,atpD,0.115896,0.242544,optimal
205,STM3866,atpG,0.115896,0.242544,optimal
206,STM3871,atpB,0.115896,0.242544,optimal
207,STM3867,atpA,0.115896,0.242544,optimal
208,STM3870,atpE,0.115896,0.242544,optimal
209,STM3868,atpH,0.115896,0.242544,optimal


In [21]:
# Save results

gene_del_annotated.to_csv(
    "simulation_3_stm_v1_single_gene_deletion_all.csv",
    index=False
)

essential_genes.to_csv(
    "simulation_3_stm_v1_essential_genes.csv",
    index=False
)

reduced_growth_genes.to_csv(
    "simulation_3_stm_v1_reduced_growth_genes.csv",
    index=False
)

In [22]:
# targeted gene knockouts 
# Check baseline FBA objective

wt_solution = stm_v1.optimize()

print("WT status:", wt_solution.status)
print("WT objective value:", wt_solution.objective_value)

WT status: optimal
WT objective value: 0.47783366076072126


In [23]:
# Define gene knockout groups using STM IDs from your model

gene_groups = {
    "sdhABCD_succinate_dehydrogenase": [
        "STM0734",  # sdhA
        "STM0735",  # sdhB
        "STM0732",  # sdhC
        "STM0733",  # sdhD
    ],
    
    "frdABCD_fumarate_reductase": [
        "STM4343",  # frdA
        "STM4342",  # frdB
        "STM4341",  # frdC
        "STM4340",  # frdD
    ],
    
    "cyoABCDE_cytochrome_bo_oxidase": [
        "STM0443",  # cyoA
        "STM0442",  # cyoB
        "STM0441",  # cyoC
        "STM0440",  # cyoD
        "STM0439",  # cyoE
    ],
    
    "cydAB_cytochrome_bd_oxidase": [
        "STM0740",  # cydA
        "STM0741",  # cydB
    ],
    
    "ldhA_lactate_fermentation": [
        "STM1647",  # ldhA
    ],
    
    "pflB_anaerobic_pyruvate_metabolism": [
        "STM0973",  # pflB
    ],
    
    "pta_ackA_acetate_production": [
        # Replace these if your STM_v1 model uses different IDs
        # Run the search cell below if either one fails
        "STM1919",  # pta, likely
        "STM1920",  # ackA, likely
    ],
}

In [24]:
for group_name, genes in gene_groups.items():
    print("\n", group_name)
    
    for gene_id in genes:
        if gene_id in [g.id for g in stm_v1.genes]:
            gene = stm_v1.genes.get_by_id(gene_id)
            print("FOUND:", gene.id, gene.name)
        else:
            print("MISSING:", gene_id)


 sdhABCD_succinate_dehydrogenase
FOUND: STM0734 sdhA
FOUND: STM0735 sdhB
FOUND: STM0732 sdhC
FOUND: STM0733 sdhD

 frdABCD_fumarate_reductase
FOUND: STM4343 frdA
FOUND: STM4342 frdB
FOUND: STM4341 frdC
FOUND: STM4340 frdD

 cyoABCDE_cytochrome_bo_oxidase
FOUND: STM0443 cyoA
FOUND: STM0442 cyoB
FOUND: STM0441 cyoC
FOUND: STM0440 cyoD
FOUND: STM0439 cyoE

 cydAB_cytochrome_bd_oxidase
FOUND: STM0740 cydA
FOUND: STM0741 cydB

 ldhA_lactate_fermentation
FOUND: STM1647 ldhA

 pflB_anaerobic_pyruvate_metabolism
FOUND: STM0973 pflB

 pta_ackA_acetate_production
MISSING: STM1919
MISSING: STM1920


In [25]:
keywords = ["pta", "ack"]

for gene in stm_v1.genes:
    text = f"{gene.id} {gene.name}".lower()
    if any(k in text for k in keywords):
        print(gene.id, gene.name)

STM2337 ackA
STM2338 pta


In [26]:
knockout_results = []

for group_name, genes in gene_groups.items():
    
    # only use genes that are actually in the model
    valid_genes = [g for g in genes if g in [gene.id for gene in stm_v1.genes]]
    missing_genes = [g for g in genes if g not in [gene.id for gene in stm_v1.genes]]
    
    with stm_v1:
        knock_out_model_genes(stm_v1, valid_genes)
        sol = stm_v1.optimize()
        
        knockout_results.append({
            "knockout_group": group_name,
            "genes_knocked_out": ";".join(valid_genes),
            "missing_genes": ";".join(missing_genes),
            "status": sol.status,
            "objective_value": sol.objective_value,
            "fraction_of_WT": sol.objective_value / wt_solution.objective_value
        })

ko_df = pd.DataFrame(knockout_results)

display(ko_df.sort_values("fraction_of_WT"))

,knockout_group,genes_knocked_out,missing_genes,status,objective_value,fraction_of_WT
2,cyoABCDE_cytochrome_bo_oxidase,STM0443;STM0442;STM0441;STM0440;STM0439,,optimal,0.418156,0.875107
0,sdhABCD_succinate_dehydrogenase,STM0734;STM0735;STM0732;STM0733,,optimal,0.457016,0.956433
1,frdABCD_fumarate_reductase,STM4343;STM4342;STM4341;STM4340,,optimal,0.477834,1.000000
3,cydAB_cytochrome_bd_oxidase,STM0740;STM0741,,optimal,0.477834,1.000000
4,ldhA_lactate_fermentation,STM1647,,optimal,0.477834,1.000000
5,pflB_anaerobic_pyruvate_metabolism,STM0973,,optimal,0.477834,1.000000
6,pta_ackA_acetate_production,,STM1919;STM1920,optimal,0.477834,1.000000


In [27]:
ko_df.to_csv("stm_v1_target_gene_group_knockouts.csv", index=False)

In [28]:
ko_df["phenotype"] = pd.cut(
    ko_df["fraction_of_WT"],
    bins=[-0.01, 0.01, 0.5, 0.9, 1.1, float("inf")],
    labels=[
        "essential_or_no_growth",
        "strong_growth_defect",
        "moderate_growth_defect",
        "near_WT",
        "higher_than_WT"
    ]
)

display(ko_df.sort_values("fraction_of_WT"))

,knockout_group,genes_knocked_out,missing_genes,status,objective_value,fraction_of_WT,phenotype
2,cyoABCDE_cytochrome_bo_oxidase,STM0443;STM0442;STM0441;STM0440;STM0439,,optimal,0.418156,0.875107,moderate_growth_defect
0,sdhABCD_succinate_dehydrogenase,STM0734;STM0735;STM0732;STM0733,,optimal,0.457016,0.956433,near_WT
1,frdABCD_fumarate_reductase,STM4343;STM4342;STM4341;STM4340,,optimal,0.477834,1.000000,near_WT
3,cydAB_cytochrome_bd_oxidase,STM0740;STM0741,,optimal,0.477834,1.000000,near_WT
4,ldhA_lactate_fermentation,STM1647,,optimal,0.477834,1.000000,near_WT
5,pflB_anaerobic_pyruvate_metabolism,STM0973,,optimal,0.477834,1.000000,near_WT
6,pta_ackA_acetate_production,,STM1919;STM1920,optimal,0.477834,1.000000,near_WT


In [29]:
mixed1090_gene_del = single_gene_deletion(mixed1090)
print(mixed1090_gene_del)

            ids        growth   status
0     {STM2991}  4.778337e-01  optimal
1     {STM0255}  4.778337e-01  optimal
2     {STM2256}  4.778337e-01  optimal
3     {STM2064}  4.778337e-01  optimal
4     {STM4453}  4.778337e-01  optimal
...         ...           ...      ...
1266  {STM0706}  4.778337e-01  optimal
1267  {STM3725}  5.058163e-12  optimal
1268  {STM1449}  4.778337e-01  optimal
1269    {s0001}  0.000000e+00  optimal
1270  {STM0226}  0.000000e+00  optimal

[1271 rows x 3 columns]


In [30]:
stmv1_gene_del["gene"] = stmv1_gene_del["ids"].apply(lambda x: list(x)[0])
stmv1_gene_del["essential"] = stmv1_gene_del["growth"] < 0.01

NameError: name 'stmv1_gene_del' is not defined

In [ ]:
## Simulations 2